In [11]:
# Step 1 — Verify GPU availability for the second DenseNet121 experiment.

import torch

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

Tesla T4


In [15]:

# Step 2 — Locate the dataset archives and the saved experiment in Google Drive.

from pathlib import Path
from tempfile import mkdtemp
from google.colab import drive

content_dir = Path("/content")
drive_root = next(
    (p for p in content_dir.glob("drive*/MyDrive")
     if (p / "ARTI502/nih_images.zip").is_file()),
    None,
)

if drive_root is None:
    mount_dir = mkdtemp(prefix="drive_check_", dir="/content")
    drive.mount(mount_dir)
    drive_root = Path(mount_dir) / "MyDrive"

project_dir = drive_root / "ARTI502"
images_zip = project_dir / "nih_images.zip"
handoff_zip = content_dir / "NIH_Dataset_Handoff.zip"
run_dir = project_dir / "DenseNet121_Experiment_2"

print("Images ZIP found:", images_zip.is_file())
print("Handoff ZIP found:", handoff_zip.is_file())
print("Best model found:", (run_dir / "best_model.pt").is_file())

Images ZIP found: True
Handoff ZIP found: False
Best model found: True


In [17]:
# Step 3 — Extract the images and load the original patient-level splits.

from shutil import copyfile
from zipfile import ZipFile
import pandas as pd

local_zip = Path("/content/nih_images.zip")
image_dir = Path("/content/nih_images")
image_dir.mkdir(exist_ok=True)

copyfile(images_zip, local_zip)

with ZipFile(local_zip) as archive:
    for entry in archive.infolist():
        filename = Path(entry.filename).name
        if filename.endswith(".png") and not filename.startswith("."):
            (image_dir / filename).write_bytes(archive.read(entry))

with ZipFile(handoff_zip) as archive:
    train_df = pd.read_csv(archive.open("train_metadata.csv"))
    val_df = pd.read_csv(archive.open("validation_metadata.csv"))
    test_df = pd.read_csv(archive.open("test_metadata.csv"))

available = {path.name for path in image_dir.glob("*.png")}

for split, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    missing = set(df["Image Index"]) - available
    print(f"{split}: {len(df):,} images | {len(missing)} missing")

Train: 78,921 images | 0 missing
Validation: 16,825 images | 0 missing
Test: 16,374 images | 0 missing


In [18]:
# Step 4 — Define the same image preprocessing used in the Mac experiment.

from torchvision import transforms

image_size = (224, 224)
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

print("Preprocessing ready.")

Preprocessing ready.


In [19]:
# Step 5 — Create the datasets and verify one training sample.

import torch
from PIL import Image
from torch.utils.data import Dataset

class ChestXrayDataset(Dataset):
    def __init__(self, metadata, image_dir, transform):
        self.filenames = metadata["Image Index"].tolist()
        self.labels = torch.tensor(
            metadata["Pneumonia"].to_numpy(), dtype=torch.float32
        )
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, index):
        filename = self.filenames[index]
        with Image.open(self.image_dir / filename) as img:
            image = self.transform(img.convert("RGB"))
        return image, self.labels[index], filename

train_dataset = ChestXrayDataset(train_df, image_dir, train_transform)
val_dataset = ChestXrayDataset(val_df, image_dir, eval_transform)
test_dataset = ChestXrayDataset(test_df, image_dir, eval_transform)

image, label, filename = train_dataset[0]
print("Filename:", filename)
print("Image shape:", tuple(image.shape))
print("Pneumonia label:", int(label.item()))

Filename: 00000001_000.png
Image shape: (3, 224, 224)
Pneumonia label: 0


In [20]:
# Step 6 — Create the data loaders and verify one training batch.

from torch.utils.data import DataLoader

batch_size = 16
generator = torch.Generator().manual_seed(42)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    generator=generator,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=0
)

test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=0
)

images, labels, filenames = next(iter(train_loader))
print("Images:", tuple(images.shape))
print("Labels:", tuple(labels.shape))
print("Filenames:", len(filenames))

Images: (16, 3, 224, 224)
Labels: (16,)
Filenames: 16


In [21]:
# Step 7 — Initialize DenseNet121 and verify its output on the Colab GPU.

from torch import nn
from torchvision.models import DenseNet121_Weights, densenet121

device = torch.device("cuda")
torch.manual_seed(42)

model = densenet121(weights=DenseNet121_Weights.DEFAULT)
model.classifier = nn.Linear(model.classifier.in_features, 1)
model = model.to(device)

model.eval()
with torch.no_grad():
    logits = model(images.to(device)).squeeze(1)

print("GPU:", torch.cuda.get_device_name(device))
print("Model output:", tuple(logits.shape))

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 71.7MB/s]


GPU: Tesla T4
Model output: (16,)


In [22]:
# Step 8 — Set up weighted binary loss using the training labels.

positive_count = int((train_df["Pneumonia"] == 1).sum())
negative_count = int((train_df["Pneumonia"] == 0).sum())

pos_weight = torch.tensor(
    [negative_count / positive_count],
    dtype=torch.float32,
    device=device,
)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"Positive training images: {positive_count:,}")
print(f"Negative training images: {negative_count:,}")
print(f"Positive class weight: {pos_weight.item():.2f}")

Positive training images: 952
Negative training images: 77,969
Positive class weight: 81.90


In [23]:
# Step 9 — Configure AdamW with the lower learning rate.

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.00003,
    weight_decay=0.01,
)

print("Optimizer:", type(optimizer).__name__)
print(f"Learning rate: {optimizer.param_groups[0]['lr']:.5f}")
print(f"Weight decay: {optimizer.param_groups[0]['weight_decay']}")

Optimizer: AdamW
Learning rate: 0.00003
Weight decay: 0.01


In [24]:
# Step 10 — Define one training epoch with a batch progress bar.

from tqdm.auto import tqdm

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = torch.zeros((), device=device)
    image_count = 0

    for images, labels, _ in tqdm(loader, desc="Training", unit="batch"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.detach() * labels.size(0)
        image_count += labels.size(0)

    return total_loss.item() / image_count

print("Training function ready.")

Training function ready.


In [25]:
# Step 11 — Calculate validation loss and average precision with a progress bar.

from sklearn.metrics import average_precision_score

@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = torch.zeros((), device=device)
    all_labels = []
    all_scores = []

    for images, labels, _ in tqdm(loader, desc="Validation", unit="batch"):
        logits = model(images.to(device)).squeeze(1)
        loss = criterion(logits, labels.to(device))

        total_loss += loss * labels.size(0)
        all_labels.append(labels)
        all_scores.append(logits.cpu())

    labels = torch.cat(all_labels).numpy()
    scores = torch.cat(all_scores).numpy()

    average_loss = total_loss.item() / len(labels)
    average_precision = average_precision_score(labels, scores)
    return average_loss, average_precision

print("Validation function ready.")

Validation function ready.


In [26]:
# Step 12 — Train DenseNet121 for three epochs and save checkpoints to Google Drive.

from pathlib import Path
from time import perf_counter

output_dir = Path("/content/drive/MyDrive/ARTI502/DenseNet121_Experiment_2")
output_dir.mkdir(parents=True, exist_ok=True)

latest_path = output_dir / "latest_checkpoint.pt"
best_path = output_dir / "best_model.pt"
history_path = output_dir / "training_history.csv"

num_epochs = 3
start_epoch = 1
best_ap = -1.0
history = []

def save_checkpoint(data, path):
    temp_path = path.with_suffix(".tmp")
    torch.save(data, temp_path)
    temp_path.replace(path)

if latest_path.exists():
    checkpoint = torch.load(latest_path, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    train_loader.generator.set_state(checkpoint["loader_rng"])
    torch.set_rng_state(checkpoint["cpu_rng"])
    history = checkpoint["history"]
    best_ap = checkpoint["best_ap"]
    start_epoch = checkpoint["epoch"] + 1
    print(f"Resuming after epoch {checkpoint['epoch']}.")

for epoch in range(start_epoch, num_epochs + 1):
    start = perf_counter()
    print(f"\nEpoch {epoch}/{num_epochs}", flush=True)

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_ap = validate_one_epoch(model, val_loader, criterion, device)
    val_ap = float(val_ap)

    if val_ap > best_ap:
        best_ap = val_ap
        save_checkpoint(model.state_dict(), best_path)

    minutes = (perf_counter() - start) / 60
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_ap": val_ap,
        "minutes": minutes,
    })

    save_checkpoint({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "best_ap": best_ap,
        "history": history,
        "loader_rng": train_loader.generator.get_state(),
        "cpu_rng": torch.get_rng_state(),
    }, latest_path)

    pd.DataFrame(history).to_csv(history_path, index=False)
    print(
        f"Epoch {epoch} | Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f} | Val AP: {val_ap:.4f} | "
        f"Time: {minutes:.1f} min",
        flush=True,
    )

print(f"\nBest validation AP: {best_ap:.4f}")


Epoch 1/3


Training:   0%|          | 0/4933 [00:00<?, ?batch/s]

Validation:   0%|          | 0/1052 [00:00<?, ?batch/s]

Epoch 1 | Train loss: 1.3109 | Val loss: 1.2665 | Val AP: 0.0290 | Time: 16.2 min

Epoch 2/3


Training:   0%|          | 0/4933 [00:00<?, ?batch/s]

Validation:   0%|          | 0/1052 [00:00<?, ?batch/s]

Epoch 2 | Train loss: 1.2467 | Val loss: 1.2612 | Val AP: 0.0279 | Time: 16.3 min

Epoch 3/3


Training:   0%|          | 0/4933 [00:00<?, ?batch/s]

Validation:   0%|          | 0/1052 [00:00<?, ?batch/s]

Epoch 3 | Train loss: 1.1629 | Val loss: 1.2725 | Val AP: 0.0314 | Time: 16.3 min

Best validation AP: 0.0314


In [27]:
# Step 13 — Locate the saved models and compare their validation records.

from pathlib import Path
import pandas as pd

for folder in sorted({Path(output_dir), Path(run_dir)}):
    model_path = folder / "best_model.pt"
    history_path = folder / "training_history.csv"

    print(f"\nFolder: {folder}")
    print("Best model found:", model_path.is_file())

    if history_path.is_file():
        results = pd.read_csv(history_path)
        best = results.loc[results["val_ap"].idxmax()]
        print(f"Best epoch: {int(best['epoch'])}")
        print(f"Best validation AP: {best['val_ap']:.4f}")
    else:
        print("Training history not found")


Folder: /content/drive/MyDrive/ARTI502/DenseNet121_Experiment_2
Best model found: True
Best epoch: 3
Best validation AP: 0.0314

Folder: /content/drive_check_0961h_e6/MyDrive/ARTI502/DenseNet121_Experiment_2
Best model found: True
Best epoch: 1
Best validation AP: 0.0322


In [28]:
# Step 14 — Verify the selected checkpoint and export its validation predictions.

selected_dir = Path(run_dir)
weights = torch.load(
    selected_dir / "best_model.pt", map_location="cpu", weights_only=True
)
model.load_state_dict(weights)
model.to(device)
model.eval()

rows = []
with torch.inference_mode():
    for images, labels, filenames in tqdm(val_loader, desc="Validation", unit="batch"):
        logits = model(images.to(device)).squeeze(1)
        scores = torch.sigmoid(logits).cpu().tolist()
        rows.extend(zip(filenames, labels.int().tolist(), scores))

predictions = pd.DataFrame(rows, columns=["Image Index", "y_true", "p_pneumonia"])
assert len(predictions) == len(val_df)
assert predictions["Image Index"].is_unique

val_ap = average_precision_score(predictions["y_true"], predictions["p_pneumonia"])
val_path = selected_dir / "densenet121_validation_predictions.csv"
predictions.to_csv(val_path, index=False)

print(f"Validation images: {len(predictions):,}")
print(f"Verified validation AP: {val_ap:.4f}")
print(f"Saved to: {val_path}")

Validation:   0%|          | 0/1052 [00:00<?, ?batch/s]

Validation images: 16,825
Verified validation AP: 0.0322
Saved to: /content/drive_check_0961h_e6/MyDrive/ARTI502/DenseNet121_Experiment_2/densenet121_validation_predictions.csv


In [29]:
# Step 15 — Evaluate the selected model on the test set and export its predictions.

from sklearn.metrics import average_precision_score, roc_auc_score

model.eval()
rows = []

with torch.inference_mode():
    for images, labels, filenames in tqdm(test_loader, desc="Test", unit="batch"):
        logits = model(images.to(device)).squeeze(1)
        scores = torch.sigmoid(logits).cpu().tolist()
        rows.extend(zip(filenames, labels.int().tolist(), scores))

test_predictions = pd.DataFrame(rows, columns=["Image Index", "y_true", "p_pneumonia"])
assert len(test_predictions) == len(test_df)
assert test_predictions["Image Index"].is_unique

labels = test_predictions["y_true"]
scores = test_predictions["p_pneumonia"]
test_ap = average_precision_score(labels, scores)
test_auc = roc_auc_score(labels, scores)

test_path = selected_dir / "densenet121_test_predictions.csv"
test_predictions.to_csv(test_path, index=False)

print(f"Test images: {len(test_predictions):,}")
print(f"Test AP: {test_ap:.4f}")
print(f"Test ROC-AUC: {test_auc:.4f}")
print(f"Saved to: {test_path}")

Test:   0%|          | 0/1024 [00:00<?, ?batch/s]

Test images: 16,374
Test AP: 0.0212
Test ROC-AUC: 0.6832
Saved to: /content/drive_check_0961h_e6/MyDrive/ARTI502/DenseNet121_Experiment_2/densenet121_test_predictions.csv


In [30]:
# Step 16 — Package the selected DenseNet121 model, predictions, metrics, and settings.

import json
import numpy as np
from zipfile import ZipFile, ZIP_DEFLATED
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    f1_score, precision_recall_curve, precision_score,
    recall_score, roc_auc_score,
)

val_path = selected_dir / "densenet121_validation_predictions.csv"
test_path = selected_dir / "densenet121_test_predictions.csv"
val = pd.read_csv(val_path)
test = pd.read_csv(test_path)

for predictions, metadata in [(val, val_df), (test, test_df)]:
    actual = predictions.set_index("Image Index")["y_true"].sort_index()
    expected = metadata.set_index("Image Index")["Pneumonia"].sort_index()
    assert actual.index.is_unique and actual.index.equals(expected.index)
    assert np.array_equal(actual.to_numpy(), expected.to_numpy())

precision, recall, thresholds = precision_recall_curve(
    val["y_true"], val["p_pneumonia"]
)
f1_values = (
    2 * precision[:-1] * recall[:-1]
    / np.maximum(precision[:-1] + recall[:-1], 1e-12)
)
threshold = float(thresholds[np.argmax(f1_values)])

rows = []
for split, predictions in [("Validation", val), ("Test", test)]:
    y = predictions["y_true"]
    score = predictions["p_pneumonia"]
    predicted = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()

    rows.append({
        "split": split, "images": len(y), "positive_images": int(y.sum()),
        "threshold": threshold, "ap_baseline": y.mean(),
        "average_precision": average_precision_score(y, score),
        "roc_auc": roc_auc_score(y, score),
        "accuracy": accuracy_score(y, predicted),
        "precision": precision_score(y, predicted, zero_division=0),
        "recall": recall_score(y, predicted),
        "f1": f1_score(y, predicted, zero_division=0),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    })

metrics = pd.DataFrame(rows)
metrics_path = selected_dir / "densenet121_metrics.csv"
metrics.to_csv(metrics_path, index=False)

settings = {
    "model": "DenseNet121, ImageNet initialization, one output logit",
    "dataset": "NIH ChestX-ray14, original shared patient-level splits",
    "target": "1 = Pneumonia label present; 0 = label absent",
    "preprocessing": "RGB, resize to 224x224, ImageNet normalization",
    "training_augmentation": "RandomHorizontalFlip(p=0.5)",
    "optimizer": "AdamW", "learning_rate": 0.00003, "weight_decay": 0.01,
    "loss": "BCEWithLogitsLoss", "pos_weight": 77969 / 952,
    "batch_size": 16, "epochs": 3, "seed": 42, "device": "Tesla T4",
    "selected_epoch": 1, "selection_metric": "Highest validation AP",
    "score_column": "p_pneumonia = sigmoid of model logit",
    "standalone_threshold": threshold,
    "threshold_rule": "Maximum validation F1, then fixed for test",
    "ensemble_instructions": (
        "Join with ResNet18 on Image Index and verify y_true agrees. "
        "Combine continuous scores. Select ensemble weights and threshold "
        "on validation, then apply both unchanged to test."
    ),
}
settings_path = selected_dir / "experiment_settings.json"
settings_path.write_text(json.dumps(settings, indent=2))

files = [
    val_path, test_path, metrics_path, settings_path,
    selected_dir / "training_history.csv",
    selected_dir / "best_model.pt",
]
zip_path = selected_dir / "DenseNet121_Colab_Selected_Handoff.zip"

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, arcname=path.name)

print(metrics.round(4).to_string(index=False))
print(f"\nHandoff ready: {zip_path.name} | {len(files)} files")

     split  images  positive_images  threshold  ap_baseline  average_precision  roc_auc  accuracy  precision  recall     f1  TP  FP  FN    TN
Validation   16825              208     0.7319       0.0124             0.0322   0.6975    0.9347     0.0468  0.2212 0.0772  46 937 162 15680
      Test   16374              193     0.7319       0.0118             0.0212   0.6832    0.9433     0.0294  0.1192 0.0472  23 758 170 15423

Handoff ready: DenseNet121_Colab_Selected_Handoff.zip | 6 files
